## Arguments

In [1]:
VIDEOS_DIR = "F:/SWAROBO/data/250723_bm_capture"
FRAME_INTERVAL = 24
MULTITHREAD = True
SERVER_MACHINE_1 = "chan@223.130.159.134"
SERVER_MACHINE_2 = "chan@223.130.137.72"
SERVER_DATA_PATH = "/home/chan/data"
SAMPLING_DENSITY = 1
LOSSLESS = True

## Setup

In [2]:
%load_ext autoreload
%autoreload 2

import os
import sys

dir_base = os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir))
sys.path.append(dir_base)

from reconstruction.utils.save_every_nth_frames_from_video import save_frames_from_videos
from reconstruction.utils.modify_registration import modify_registration

OUTPUT_DIR = os.path.realpath(os.path.join(VIDEOS_DIR, "data_"+os.path.basename(VIDEOS_DIR)))
VIDEOS_DIR = os.path.realpath(VIDEOS_DIR)
IMAGES_DIR = os.path.join(VIDEOS_DIR, "images")

def run(command):
    print(command)
    os.system(command)

## 1. Videos -> Images

In [3]:
save_frames_from_videos(
    videos_folder_path=VIDEOS_DIR,
    frame_interval=FRAME_INTERVAL,
    multithread=MULTITHREAD,
    lossless=LOSSLESS,
)

Press 'q' to stop the process.


## 2. SFM

In [ ]:
run(f"""{
    os.path.join(
        dir_base,
        "reality_scan",
        "RS_SFM.BAT"
    )
} {VIDEOS_DIR}""")

d:\Users\chany\Documents\SWAROBO\reality_capture\RC_SFM.BAT D:\Users\chany\Downloads\250403_incheon_car_dome


## 3. Modify registration

In [13]:
modify_registration(OUTPUT_DIR)

FileNotFoundError: [WinError 2] The system cannot find the file specified: 'D:\\Users\\chany\\Downloads\\250411_youngwol_mirrorless\\data_250411_youngwol_mirrorless\\images\\transforms.json' -> 'D:\\Users\\chany\\Downloads\\250411_youngwol_mirrorless\\data_250411_youngwol_mirrorless\\transforms.json'

## 4. Transfer to server

In [14]:
for server_machine, folder_dir in [(SERVER_MACHINE_1, OUTPUT_DIR), (SERVER_MACHINE_2, IMAGES_DIR)]:
    SERVER_DATA_ADDED_PATH = SERVER_DATA_PATH+"/"+os.path.basename(OUTPUT_DIR)
    run(f"ssh {server_machine} mkdir -p {SERVER_DATA_ADDED_PATH}")
    run(f"scp -vr {folder_dir}/* {server_machine}:{SERVER_DATA_ADDED_PATH}/")
    run(f"scp -vr {os.path.join(dir_base, 'reconstruction', 'scripts', 'server-scripts')} {server_machine}:/home/chan/")
    break

ssh chan@223.130.159.134 mkdir -p /home/chan/data/data_250411_youngwol_mirrorless
scp -vr D:\Users\chany\Downloads\250411_youngwol_mirrorless\data_250411_youngwol_mirrorless/* chan@223.130.159.134:/home/chan/data/data_250411_youngwol_mirrorless/
scp -vr d:\Users\chany\Documents\SWAROBO\reconstruction\scripts\server-scripts chan@223.130.159.134:/home/chan/


## 5. Start training process

In [15]:
for server_machine, colmap in [(SERVER_MACHINE_1, "False"), (SERVER_MACHINE_2, "True")]:
    command = f"ssh {server_machine} bash /home/chan/server-scripts/run.bash \
        {SERVER_DATA_ADDED_PATH} \
        {colmap} \
        {SAMPLING_DENSITY}"
    run(command)
    break

ssh chan@223.130.159.134 bash /home/chan/server-scripts/run.bash         /home/chan/data/data_250411_youngwol_mirrorless         False         1
